In [6]:
import requests
from bs4 import BeautifulSoup

# ============================================================
# 厚労省ページから雇用保険料率の「更新日」を取得する
# 料率が変わったことを検知するシステム
# ============================================================

url = "https://www.mhlw.go.jp/stf/seisakunitsuite/bunya/0000108634.html"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

response = requests.get(url, headers=headers)
print(f"ステータスコード：{response.status_code}")

soup = BeautifulSoup(response.content, "html.parser")

# ページ内のテキストから料率情報を探す
text = soup.get_text()
lines = [l.strip() for l in text.splitlines() if l.strip()]

print("\n料率に関連するキーワードを含む行：")
for line in lines:
    if any(kw in line for kw in ["1,000", "料率", "令和", "保険料率"]):
        print(f"  {line}")

ステータスコード：200

料率に関連するキーワードを含む行：
  雇用保険料率について ｜厚生労働省
  雇用保険料率について
  雇用保険料率について
  各年度の雇用保険料率
  令和８年度の雇用保険料率
  令和８年度の雇用保険料率について［143KB］
  令和７年度の雇用保険料率
  令和7年度の雇用保険料率について［342KB］
  令和６年度の雇用保険料率
  令和6年度の雇用保険料率について［98KB］
  令和５年度の雇用保険料率
  令和５年度の雇用保険料率について［316KB］
  令和４年度の雇用保険料率
  令和４年度の雇用保険料率について［110KB］
  令和３年度の雇用保険料率
  令和３年度の雇用保険料率について［106KB］


In [7]:
# PDFのリンクを取得する
links = soup.find_all("a", href=True)

print("雇用保険料率のPDFリンク一覧：")
for link in links:
    href = link.get("href", "")
    text = link.get_text(strip=True)
    if "雇用保険料率" in text and ("PDF" in text or "KB" in text or ".pdf" in href.lower()):
        full_url = href if href.startswith("http") else f"https://www.mhlw.go.jp{href}"
        print(f"  {text}")
        print(f"  → {full_url}")
        print()

雇用保険料率のPDFリンク一覧：
  令和８年度の雇用保険料率について［143KB］
  → https://www.mhlw.go.jp/content/001672589.pdf

  令和7年度の雇用保険料率について［342KB］
  → https://www.mhlw.go.jp/content/001401966.pdf

  令和6年度の雇用保険料率について［98KB］
  → https://www.mhlw.go.jp/content/001211914.pdf

  令和５年度の雇用保険料率について［316KB］
  → https://www.mhlw.go.jp/content/001050206.pdf

  令和４年度の雇用保険料率について［110KB］
  → https://www.mhlw.go.jp/content/000921550.pdf

  令和３年度の雇用保険料率について［106KB］
  → https://www.mhlw.go.jp/content/000739455.pdf



In [8]:
# pdfplumberをインストール
!pip install pdfplumber -q
print("✅ インストール完了")

✅ インストール完了


In [9]:
import pdfplumber
import io

# 令和8年度の雇用保険料率PDFをダウンロード
pdf_url = "https://www.mhlw.go.jp/content/001672589.pdf"
headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}

response = requests.get(pdf_url, headers=headers)
print(f"ステータスコード：{response.status_code}")
print(f"ファイルサイズ：{len(response.content):,} bytes")

# PDFをメモリ上で開く
with pdfplumber.open(io.BytesIO(response.content)) as pdf:
    print(f"\nページ数：{len(pdf.pages)}ページ")
    page = pdf.pages[0]
    text = page.extract_text()
    print("\n--- PDFテキスト ---")
    print(text)

ステータスコード：200
ファイルサイズ：146,295 bytes

ページ数：1ページ

--- PDFテキスト ---
事業主・被保険者の皆さまへ
令和８（202６）年度 雇用保険料率のご案内
令和８（202６）年４月１日から令和９（202７）年３月31日までの雇用保険料率は以下
のとおりです。
• 失業等給付等の保険料率は、労働者負担・事業主負担ともに5/1,000に変更に
なります（農林水産・清酒製造の事業及び建設の事業は6/1,000に変更になり
ます。）。
• 雇用保険二事業の保険料率（事業主のみ負担）は、引き続き3.5/1,000です（建
設の事業は4.5/1,000です。）。
＜令和８年度の雇用保険料率＞
（赤字は変更部分）
① ②
負担者
労働者負担 事業主負担 ①＋②
（失業等給付・ 失業等給付・ 雇用保険二事業 雇用保険料率
育児休業給付の 育児休業給付の
事業の種類 保険料率のみ） 保険料率 の保険料率
一般の事業 5/1,000 8.5/1,000 5/1,000 3.5/1,000 13.5/1,000
（令和７年度） 5.5/1,000 9/1,000 5.5/1,000 3.5/1,000 14.5/1,000
※
農林水産・
6/1,000 9.5/1,000 6/1,000 3.5/1,000 15.5/1,000
清酒製造の事業
（令和７年度）
6.5/1,000 10/1,000 6.5/1,000 3.5/1,000 16.5/1,000
建設の事業 6/1,000 10.5/1,000 6/1,000 4.5/1,000 16.5/1,000
（令和７年度）
6.5/1,000 11/1,000 6.5/1,000 4.5/1,000 17.5/1,000
（枠内の下段は令和７年４月～令和８年３月の雇用保険料率）
※ 園芸サービス、牛馬の育成、酪農、養鶏、養豚、内水面養殖および特定の船員を雇用する事業については一
般の事業の率が適用されます。
LL0８０３１２保01


In [10]:
import re

# PDFテキストから料率を自動抽出する
def extract_koyo_rates(text):
    """雇用保険料率をPDFテキストから自動抽出する"""

    rates = {}

    # 業種ごとの料率パターンを検索
    # 「X/1,000」という形式を探す
    patterns = {
        "一般":         r"一般の事業\s+(\d+(?:\.\d+)?)/1,000",
        "農林水産":     r"農林水産・\s*清酒製造の事業\s+(\d+(?:\.\d+)?)/1,000",
        "建設":         r"建設の事業\s+(\d+(?:\.\d+)?)/1,000",
    }

    for gyoshu, pattern in patterns.items():
        match = re.search(pattern, text)
        if match:
            rate = float(match.group(1)) / 1000
            rates[gyoshu] = rate
            print(f"✅ {gyoshu}：{match.group(1)}/1,000 = {rate*100:.1f}%（労働者負担）")
        else:
            print(f"⚠️ {gyoshu}：パターンが見つかりませんでした")

    return rates

# 抽出実行
print("=== 令和8年度 雇用保険料率（労働者負担）===")
koyo_rates = extract_koyo_rates(text)

print("\n=== 前年度との比較 ===")
old_rates = {"一般": 0.0055, "農林水産": 0.0065, "建設": 0.0065}
for gyoshu, new_rate in koyo_rates.items():
    old_rate = old_rates.get(gyoshu, 0)
    diff = (new_rate - old_rate) * 1000
    mark = "↓" if diff < 0 else "↑" if diff > 0 else "→"
    print(f"  {gyoshu}：{old_rate*1000:.1f}/1,000 {mark} {new_rate*1000:.1f}/1,000")

=== 令和8年度 雇用保険料率（労働者負担）===
✅ 一般：5/1,000 = 0.5%（労働者負担）
⚠️ 農林水産：パターンが見つかりませんでした
✅ 建設：6/1,000 = 0.6%（労働者負担）

=== 前年度との比較 ===
  一般：5.5/1,000 ↓ 5.0/1,000
  建設：6.5/1,000 ↓ 6.0/1,000


In [12]:
#googleドライブに接続
from google.colab import drive
drive.mount('/content/drive')

import re

def extract_koyo_rates(text):
    """雇用保険料率をPDFテキストから自動抽出する（改行対応版）"""

    # 改行を含む可能性があるため、テキストを1行に結合
    text_oneline = " ".join(text.splitlines())

    rates = {}

    patterns = {
        "一般":     r"一般の事業\s+(\d+(?:\.\d+)?)/1,000",
        "農林水産": r"農林水産[・\s]+清酒製造の事業\s+(\d+(?:\.\d+)?)/1,000",
        "建設":     r"建設の事業\s+(\d+(?:\.\d+)?)/1,000",
    }

    for gyoshu, pattern in patterns.items():
        match = re.search(pattern, text_oneline)
        if match:
            rate = float(match.group(1)) / 1000
            rates[gyoshu] = rate
            print(f"✅ {gyoshu}：{match.group(1)}/1,000 = {rate*100:.2f}%（労働者負担）")
        else:
            print(f"⚠️ {gyoshu}：パターンが見つかりませんでした")

    return rates

print("=== 令和8年度 雇用保険料率（労働者負担）===")
koyo_rates = extract_koyo_rates(text)

print("\n=== 前年度との比較 ===")
old_rates = {"一般": 0.0055, "農林水産": 0.0065, "建設": 0.0065}
for gyoshu, new_rate in koyo_rates.items():
    old_rate = old_rates.get(gyoshu, 0)
    diff = (new_rate - old_rate) * 1000
    mark = "↓" if diff < 0 else "↑" if diff > 0 else "→"
    print(f"  {gyoshu}：{old_rate*1000:.1f}/1,000 {mark} {new_rate*1000:.1f}/1,000")

# JSONファイルに保存（来年比較用）
import json
from datetime import datetime

save_data = {
    "年度": "令和8年度",
    "取得日": datetime.now().strftime("%Y/%m/%d"),
    "雇用保険料率_労働者負担": koyo_rates
}

save_path = "/content/drive/MyDrive/Colab Notebooks/雇用保険料率_令和8年度.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(save_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ JSONファイルに保存しました")
print(f"保存先：{save_path}")

Mounted at /content/drive
=== 令和8年度 雇用保険料率（労働者負担）===
✅ 一般：5/1,000 = 0.50%（労働者負担）
⚠️ 農林水産：パターンが見つかりませんでした
✅ 建設：6/1,000 = 0.60%（労働者負担）

=== 前年度との比較 ===
  一般：5.5/1,000 ↓ 5.0/1,000
  建設：6.5/1,000 ↓ 6.0/1,000

✅ JSONファイルに保存しました
保存先：/content/drive/MyDrive/Colab Notebooks/雇用保険料率_令和8年度.json


In [13]:
pip show pdfplumber

Name: pdfplumber
Version: 0.11.9
Summary: Plumb a PDF for detailed information about each char, rectangle, and line.
Home-page: https://github.com/jsvine/pdfplumber
Author: Jeremy Singer-Vine
Author-email: jsvine@gmail.com
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: pdfminer.six, Pillow, pypdfium2
Required-by: 


In [14]:
import re

def extract_koyo_rates_v2(text):
    """行単位で解析して料率を抽出する（改良版）"""

    lines = text.splitlines()
    rates = {}

    # 業種名を見つけたら次の数字行を料率として取得する
    target_gyoshu = None

    for i, line in enumerate(lines):
        line = line.strip()

        # 業種名の検出
        if "一般の事業" in line and "令和" not in line:
            target_gyoshu = "一般"
        elif "農林水産" in line:
            target_gyoshu = "農林水産"
        elif "建設の事業" in line and "令和" not in line:
            target_gyoshu = "建設"

        # 料率の検出（X/1,000 の形式）
        if target_gyoshu and re.search(r"^\d+/1,000", line):
            match = re.match(r"^(\d+(?:\.\d+)?)/1,000", line)
            if match and target_gyoshu not in rates:
                rate = float(match.group(1)) / 1000
                rates[target_gyoshu] = rate
                print(f"✅ {target_gyoshu}：{match.group(1)}/1,000 = {rate*100:.2f}%")
                target_gyoshu = None  # 取得済みにする

    return rates

print("=== 令和8年度 雇用保険料率（労働者負担）===")
koyo_rates_v2 = extract_koyo_rates_v2(text)

print("\n=== 前年度との比較 ===")
old_rates = {"一般": 0.0055, "農林水産": 0.0065, "建設": 0.0065}
for gyoshu, new_rate in koyo_rates_v2.items():
    old_rate = old_rates.get(gyoshu, 0)
    diff = (new_rate - old_rate) * 1000
    mark = "↓" if diff < 0 else "↑" if diff > 0 else "→"
    print(f"  {gyoshu}：{old_rate*1000:.1f}/1,000 {mark} {new_rate*1000:.1f}/1,000")

# JSONに保存
import json
from datetime import datetime

save_data = {
    "年度": "令和8年度",
    "取得日": datetime.now().strftime("%Y/%m/%d"),
    "雇用保険料率_労働者負担": koyo_rates_v2
}

save_path = "/content/drive/MyDrive/Colab Notebooks/雇用保険料率_令和8年度.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(save_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ JSONファイルに保存しました")

=== 令和8年度 雇用保険料率（労働者負担）===
✅ 農林水産：6/1,000 = 0.60%

=== 前年度との比較 ===
  農林水産：6.5/1,000 ↓ 6.0/1,000

✅ JSONファイルに保存しました


In [15]:
import re
import json
from datetime import datetime

def extract_koyo_rates_final(text):
    """雇用保険料率を抽出する（最終版）"""

    lines = text.splitlines()
    rates = {}

    # ============================================================
    # 方法①：正規表現パターンで一括検索（一般・建設向き）
    # ============================================================
    text_oneline = " ".join(lines)

    patterns = {
        "一般":     r"一般の事業\s+(\d+(?:\.\d+)?)/1,000",
        "建設":     r"建設の事業\s+(\d+(?:\.\d+)?)/1,000",
    }

    for gyoshu, pattern in patterns.items():
        match = re.search(pattern, text_oneline)
        if match:
            rate = float(match.group(1)) / 1000
            rates[gyoshu] = rate

    # ============================================================
    # 方法②：行単位で解析（農林水産向き・料率が間に挟まれるため）
    # ============================================================
    target_gyoshu = None

    for line in lines:
        line = line.strip()

        if "農林水産" in line:
            target_gyoshu = "農林水産"

        if target_gyoshu == "農林水産" and re.search(r"^\d+/1,000", line):
            match = re.match(r"^(\d+(?:\.\d+)?)/1,000", line)
            if match and "農林水産" not in rates:
                rate = float(match.group(1)) / 1000
                rates["農林水産"] = rate
                target_gyoshu = None

    # ============================================================
    # 結果表示
    # ============================================================
    print("=== 令和8年度 雇用保険料率（労働者負担）===")
    all_gyoshu = ["一般", "農林水産", "建設"]
    for gyoshu in all_gyoshu:
        if gyoshu in rates:
            rate = rates[gyoshu]
            print(f"✅ {gyoshu}：{int(rate*1000)}/1,000 = {rate*100:.2f}%（労働者負担）")
        else:
            print(f"⚠️ {gyoshu}：パターンが見つかりませんでした")

    # ============================================================
    # 前年度との比較
    # ============================================================
    print("\n=== 前年度との比較 ===")
    old_rates = {"一般": 0.0055, "農林水産": 0.0065, "建設": 0.0065}
    for gyoshu in all_gyoshu:
        if gyoshu in rates:
            new_rate = rates[gyoshu]
            old_rate = old_rates.get(gyoshu, 0)
            diff = (new_rate - old_rate) * 1000
            mark = "↓" if diff < 0 else "↑" if diff > 0 else "→"
            print(f"  {gyoshu}：{old_rate*1000:.1f}/1,000 {mark} {new_rate*1000:.1f}/1,000")

    return rates

# 実行
koyo_rates_final = extract_koyo_rates_final(text)

# JSONに保存
save_data = {
    "年度": "令和8年度",
    "取得日": datetime.now().strftime("%Y/%m/%d"),
    "雇用保険料率_労働者負担": koyo_rates_final
}

save_path = "/content/drive/MyDrive/Colab Notebooks/雇用保険料率_令和8年度.json"
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(save_data, f, ensure_ascii=False, indent=2)

print(f"\n✅ JSONファイルに保存しました")
print(f"保存先：{save_path}")

=== 令和8年度 雇用保険料率（労働者負担）===
✅ 一般：5/1,000 = 0.50%（労働者負担）
✅ 農林水産：6/1,000 = 0.60%（労働者負担）
✅ 建設：6/1,000 = 0.60%（労働者負担）

=== 前年度との比較 ===
  一般：5.5/1,000 ↓ 5.0/1,000
  農林水産：6.5/1,000 ↓ 6.0/1,000
  建設：6.5/1,000 ↓ 6.0/1,000

✅ JSONファイルに保存しました
保存先：/content/drive/MyDrive/Colab Notebooks/雇用保険料率_令和8年度.json
